In [12]:
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output

# ==========================================
# 1. إعداد البيانات ومحرك الحسابات (نفس الكود السابق بالضبط)
# ==========================================
agents_df = pd.DataFrame({
    'agent_id': ['A01', 'A02', 'A03', 'A04', 'A05'],
    'agent_name': ['Ahmed Saad', 'Sarah Ali', 'Omar Hassan', 'Laila Mahmoud', 'Khaled Idriss'],
    'team': ['Travel Gate - Sales', 'Travel Gate - Support', 'Travel Gate - Sales', 'Travel Gate - Sales', 'Travel Gate - Support'],
})

performance_df = pd.DataFrame({
    'agent_id': ['A01', 'A02', 'A03', 'A04', 'A05'],
    'avg_response_time': [12.5, 8.2, 18.0, 14.9, 11.1],
    'quality_score': [4.2, 4.8, 3.5, 4.1, 4.5],
    'conversion_rate': [0.32, 0.36, 0.22, 0.31, 0.28],
    'punctuality_rate': [0.95, 0.98, 0.88, 0.92, 0.96],
    'dev_hours': [3.8, 4.2, 2.1, 3.5, 3.9],
    'csat_score': [4.3, 4.9, 3.8, 4.2, 4.6],
    'tl_score': [4.5, 5.0, 3.0, 4.0, 4.8]
})

def calculate_performance(row):
    weights = {'speed': 0.10, 'quality': 0.20, 'conv': 0.30, 'punct': 0.10, 'dev': 0.10, 'csat': 0.15, 'tl': 0.05}
    scores = {}
    scores['speed'] = max(0, (15 - row['avg_response_time']) / 15) * 100 * weights['speed']
    scores['quality'] = (row['quality_score'] / 5) * 100 * weights['quality']
    
    cr = row['conversion_rate']
    if cr >= 0.35: multiplier = 1.2
    elif cr >= 0.30: multiplier = 1.0
    elif cr >= 0.25: multiplier = 0.8
    else: multiplier = (cr / 0.25) * 0.7
    scores['conv'] = multiplier * 100 * weights['conv']
    
    scores['punct'] = (row['punctuality_rate'] / 0.90) * 100 * weights['punct']
    scores['dev'] = (row['dev_hours'] / 3.5) * 100 * weights['dev']
    scores['csat'] = (row['csat_score'] / 5) * 100 * weights['csat']
    scores['tl'] = (row['tl_score'] / 5) * 100 * weights['tl']
    
    return sum(scores.values()), scores

df = pd.merge(agents_df, performance_df, on='agent_id')
df[['final_score', 'breakdown']] = df.apply(lambda r: calculate_performance(r), axis=1, result_type='expand')

# ==========================================
# 2. إعداد الرسم البياني (Plotly Figure)
# ==========================================
target_line_value = 85.0

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['agent_name'], y=df['final_score'],
    marker=dict(color='#2c3e50', line=dict(width=1, color='black')),
    text=[f"{s:.1f}%" for s in df['final_score']], textposition='outside',
    customdata=df['agent_name'] # لتمرير اسم الموظف عند النقر
))

fig.add_hline(y=target_line_value, line_dash="dash", line_color="red", 
              annotation_text=f"Company Target ({target_line_value}%)", annotation_position="top right")

fig.update_layout(
    title="<b>Agents Monthly Performance Overview</b>",
    yaxis=dict(title="Performance Score (%)", range=[0, 115]),
    plot_bgcolor='white', margin=dict(t=50, b=50, l=50, r=50),
    clickmode='event+select'
)

# ==========================================
# 3. بناء تطبيق Dash للويب
# ==========================================
app = Dash(__name__)

# تصميم صفحة الويب
app.layout = html.Div(style={'fontFamily': 'sans-serif', 'maxWidth': '1000px', 'margin': 'auto'}, children=[
    html.H1("لوحة تحكم أداء الموظفين - Travel Gate", style={'textAlign': 'center', 'color': '#2c3e50'}),
    dcc.Graph(id='performance-chart', figure=fig),
    html.Div(id='detail-panel', style={'marginTop': '20px'}) # المكان اللي هيظهر فيه التقرير
])

# دالة التفاعل (لما تضغط على الرسمة، ايه اللي يحصل؟)
@app.callback(
    Output('detail-panel', 'children'),
    Input('performance-chart', 'clickData')
)
def update_details(clickData):
    if clickData is None:
        return html.Div("اضغط على أي عمود لعرض التفاصيل...", style={'textAlign': 'center', 'color': 'gray', 'padding': '20px'})
    
    # استخراج اسم الموظف من الضغطة
    agent_name = clickData['points'][0]['customdata']
    data = df[df['agent_name'] == agent_name].iloc[0]
    br = data['breakdown']
    
    # لون النتيجة (أخضر لو عدى التارجت، أحمر لو لأ)
    score_color = 'green' if data['final_score'] >= target_line_value else 'red'
    
    # بناء التقرير
    return html.Div(style={'border': '2px solid #2c3e50', 'padding': '20px', 'borderRadius': '10px', 'background': '#f9f9f9', 'direction': 'rtl'}, children=[
        html.H2(f"📋 تقرير الأداء التفصيلي: {data['agent_name']}", style={'color': '#2c3e50', 'marginTop': '0'}),
        html.P([html.B("الفريق: "), data['team'], " | ", html.B("كود الموظف: "), data['agent_id']]),
        html.Hr(),
        html.Div(style={'textAlign': 'center', 'marginBottom': '20px'}, children=[
            html.Small("النتيجة النهائية"),
            html.Br(),
            html.Strong(f"{data['final_score']:.1f}%", style={'fontSize': '32px', 'color': score_color})
        ]),
        html.Table(style={'width': '100%', 'borderCollapse': 'collapse', 'textAlign': 'right'}, children=[
            html.Tr(style={'background': '#2c3e50', 'color': 'white'}, children=[
                html.Th("المؤشر", style={'padding': '10px'}), html.Th("القيمة المحققة", style={'padding': '10px'}), html.Th("النتيجة الموزونة", style={'padding': '10px'})
            ]),
            html.Tr([html.Td("سرعة الرد"), html.Td(f"{data['avg_response_time']} دقيقة"), html.Td(f"{br['speed']:.1f}%")], style={'padding': '8px'}),
            html.Tr([html.Td("جودة المحادثات"), html.Td(f"{data['quality_score']}/5"), html.Td(f"{br['quality']:.1f}%")], style={'background': '#eee', 'padding': '8px'}),
            html.Tr([html.Td("معدل التحويل"), html.Td(f"{data['conversion_rate']*100:.1f}%"), html.Td(f"{br['conv']:.1f}%")], style={'padding': '8px'}),
            html.Tr([html.Td("الالتزام"), html.Td(f"{data['punctuality_rate']*100:.1f}%"), html.Td(f"{br['punct']:.1f}%")], style={'background': '#eee', 'padding': '8px'}),
            html.Tr([html.Td("ساعات التطوير"), html.Td(f"{data['dev_hours']} ساعة"), html.Td(f"{br['dev']:.1f}%")], style={'padding': '8px'}),
            html.Tr([html.Td("رضا العملاء"), html.Td(f"{data['csat_score']}/5"), html.Td(f"{br['csat']:.1f}%")], style={'background': '#eee', 'padding': '8px'}),
            html.Tr([html.Td("تقييم التيم ليدر"), html.Td(f"{data['tl_score']}/5"), html.Td(f"{br['tl']:.1f}%")], style={'padding': '8px'})
        ])
    ])

# تشغيل السيرفر المحلي
if __name__ == '__main__':
    app.run(debug=True)

In [11]:
import pandas as pd
import plotly.graph_objects as go

# 1. الداتا والحسابات (زي ما هي بالظبط من كودك)
agents_df = pd.DataFrame({'agent_id': ['A01', 'A02', 'A03', 'A04', 'A05'], 'agent_name': ['Ahmed Saad', 'Sarah Ali', 'Omar Hassan', 'Laila Mahmoud', 'Khaled Idriss'], 'team': ['Travel Gate - Sales', 'Travel Gate - Support', 'Travel Gate - Sales', 'Travel Gate - Sales', 'Travel Gate - Support']})
performance_df = pd.DataFrame({'agent_id': ['A01', 'A02', 'A03', 'A04', 'A05'], 'avg_response_time': [12.5, 8.2, 18.0, 14.9, 11.1], 'quality_score': [4.2, 4.8, 3.5, 4.1, 4.5], 'conversion_rate': [0.32, 0.36, 0.22, 0.31, 0.28], 'punctuality_rate': [0.95, 0.98, 0.88, 0.92, 0.96], 'dev_hours': [3.8, 4.2, 2.1, 3.5, 3.9], 'csat_score': [4.3, 4.9, 3.8, 4.2, 4.6], 'tl_score': [4.5, 5.0, 3.0, 4.0, 4.8]})

def calculate_performance(row):
    weights = {'speed': 0.10, 'quality': 0.20, 'conv': 0.30, 'punct': 0.10, 'dev': 0.10, 'csat': 0.15, 'tl': 0.05}
    scores = {'speed': max(0, (15 - row['avg_response_time']) / 15) * 100 * weights['speed'],
              'quality': (row['quality_score'] / 5) * 100 * weights['quality']}
    cr = row['conversion_rate']
    multiplier = 1.2 if cr >= 0.35 else 1.0 if cr >= 0.30 else 0.8 if cr >= 0.25 else (cr / 0.25) * 0.7
    scores['conv'] = multiplier * 100 * weights['conv']
    scores['punct'] = (row['punctuality_rate'] / 0.90) * 100 * weights['punct']
    scores['dev'] = (row['dev_hours'] / 3.5) * 100 * weights['dev']
    scores['csat'] = (row['csat_score'] / 5) * 100 * weights['csat']
    scores['tl'] = (row['tl_score'] / 5) * 100 * weights['tl']
    return sum(scores.values()), scores

df = pd.merge(agents_df, performance_df, on='agent_id')
df[['final_score', 'breakdown']] = df.apply(lambda r: calculate_performance(r), axis=1, result_type='expand')

# 2. تجهيز النص اللي هيظهر لما يقف بالماوس (Hover Text)
hover_texts = []
for index, row in df.iterrows():
    br = row['breakdown']
    hover_text = f"""
    <b>{row['agent_name']}</b><br>
    Team: {row['team']}<br>
    <hr>
    <b>Final Score: {row['final_score']:.1f}%</b><br>
    <hr>
    Speed: {br['speed']:.1f}%<br>
    Quality: {br['quality']:.1f}%<br>
    Conversion: {br['conv']:.1f}%<br>
    Punctuality: {br['punct']:.1f}%<br>
    Dev Hours: {br['dev']:.1f}%<br>
    CSAT: {br['csat']:.1f}%<br>
    TL Score: {br['tl']:.1f}%
    """
    hover_texts.append(hover_text)

# 3. رسم الشارت وحفظه كملف HTML
target_line_value = 85.0
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['agent_name'], y=df['final_score'],
    marker=dict(color='#2c3e50', line=dict(width=1, color='black')),
    text=[f"{s:.1f}%" for s in df['final_score']], textposition='outside',
    hoverinfo="text", hovertext=hover_texts # ربط نص الـ Hover بالعمود
))

fig.add_hline(y=target_line_value, line_dash="dash", line_color="red", 
              annotation_text=f"Company Target ({target_line_value}%)", annotation_position="top right")

fig.update_layout(title="<b>Travel Gate - Agents Performance Overivew</b>", yaxis=dict(range=[0, 115]), plot_bgcolor='white')

# السطر السحري لحفظ الملف
fig.write_html("HR_Report.html")
print("✅ تم إنشاء ملف HR_Report.html بنجاح! يمكنك إرساله للمدير الآن.")

✅ تم إنشاء ملف HR_Report.html بنجاح! يمكنك إرساله للمدير الآن.
